<a href="https://colab.research.google.com/github/egaeuni/2026-1-DL-Final-Project/blob/main/DL_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna

In [ ]:
# ---------------------------------------------------------
# 0. 환경 설정
# ---------------------------------------------------------
!pip install optuna holidays lightgbm -q

import os, random, copy, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
import joblib
import lightgbm as lgb

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

data_dir = '/content/drive/MyDrive/DeepLearning_Data/Final_Project/'

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
drive.mount('/content/drive')

DIR_PATH = '/content/drive/MyDrive/DeepLearning_Data/Final_Project'
train_path = os.path.join(DIR_PATH, 'train.csv')
test_path = os.path.join(DIR_PATH, 'test.csv')

raw_train_df = pd.read_csv(train_path)
raw_test_df = pd.read_csv(test_path)

print(f"최초 로드 - Train: {raw_train_df.shape} | Test: {raw_test_df.shape}")

In [ ]:
# ---------------------------------------------------------
# 2. 파생 변수 생성
# ---------------------------------------------------------
def add_kma_warning_features(df):
    """
    [기상청 특보 파생 변수 생성 함수]
    온도와 습도를 기반으로 한파/폭염 특보 발효 여부 및 연속 발생 일수를 계산
    """
    df = df.copy()
    T, H, V = df['Temp'], df['Humi'], df['Wind']
    
    # ── 1. 열지수(HeatIndex) 계산 ──
    # 기온 27도, 습도 40% 이상일 때 체감하는 더위를 수치화
    heat_idx_raw = (-0.2442 + 0.55399*T + 0.45535*H - 0.0022*H**2 + 0.00278*T*H)
    df['HeatIndex'] = np.where((T >= 27) & (H >= 40), heat_idx_raw, T)

    # 일별 요약 데이터 생성 (특보는 일 단위로 발효되므로)
    daily = df.groupby(['Year','Month','Day']).agg(
        LowTemp_daily      = ('LowTemp',    'first'),
        AverageTemp_daily  = ('AverageTemp','first'),
        MaxHeatIndex_daily = ('HeatIndex',  'max'),
    ).reset_index()

    # 전일 대비 최저기온 하강폭 계산
    daily['LowTemp_prev1d']  = daily['LowTemp_daily'].shift(1)
    daily['LowTemp_drop_1d'] = daily['LowTemp_prev1d'] - daily['LowTemp_daily']
    daily['Below_avg3'] = ((daily['AverageTemp_daily'] - daily['LowTemp_daily']) >= 3).astype(int)

    # ── 2. 한파 주의보/경보 조건 ──
    # 10월~4월 사이에 기온이 급격히 떨어지거나, 영하권(-12도, -15도)이 지속될 때
    cold_season   = daily['Month'].isin([10,11,12,1,2,3,4])
    base_cond1    = (daily['LowTemp_daily'] <= 3) & (daily['Below_avg3'] == 1)
    
    cold_watch_c1 = cold_season & base_cond1 & (daily['LowTemp_drop_1d'] >= 10)
    cold_warn_c1  = cold_season & base_cond1 & (daily['LowTemp_drop_1d'] >= 15)
    
    below_neg12   = (daily['LowTemp_daily'] <= -12).astype(int)
    below_neg15   = (daily['LowTemp_daily'] <= -15).astype(int)
    cold_watch_c2 = cold_season & (below_neg12.rolling(2, min_periods=2).min() == 1)
    cold_warn_c2  = cold_season & (below_neg15.rolling(2, min_periods=2).min() == 1)

    daily['Coldwave_Watch']   = (cold_watch_c1 | cold_watch_c2).astype(int)
    daily['Coldwave_Warning'] = (cold_warn_c1  | cold_warn_c2 ).astype(int)
    
    # 주의보/경보 중첩 방지 및 단계화(Level)
    daily['Coldwave_Watch']   = np.maximum(daily['Coldwave_Watch'], daily['Coldwave_Warning'])
    daily['Coldwave_Level']   = daily['Coldwave_Watch'] + daily['Coldwave_Warning']

    # ── 3. 폭염 주의보/경보 조건 ──
    # 일 최고 열지수가 33도(주의보), 35도(경보) 이상이 이틀 이상 지속될 때
    above_33 = (daily['MaxHeatIndex_daily'] >= 33).astype(int)
    above_35 = (daily['MaxHeatIndex_daily'] >= 35).astype(int)
    
    daily['Heatwave_Watch']   = (above_33.rolling(2, min_periods=2).min() == 1).astype(int)
    daily['Heatwave_Warning'] = (above_35.rolling(2, min_periods=2).min() == 1).astype(int)
    
    # 주의보/경보 중첩 방지 및 단계화(Level)
    daily['Heatwave_Watch']   = np.maximum(daily['Heatwave_Watch'], daily['Heatwave_Warning'])
    daily['Heatwave_Level']   = daily['Heatwave_Watch'] + daily['Heatwave_Warning']

    # 일별 계산 결과를 원본 시간 단위 데이터에 병합
    merge_cols = ['Year','Month','Day',
                  'LowTemp_prev1d','LowTemp_drop_1d','Below_avg3','MaxHeatIndex_daily',
                  'Coldwave_Watch','Coldwave_Warning','Coldwave_Level',
                  'Heatwave_Watch','Heatwave_Warning','Heatwave_Level']
    df = df.merge(daily[merge_cols], on=['Year','Month','Day'], how='left')

    # ── 4. 특보 연속 발생 일수(Streak) 계산 ──
    df['Coldwave_streak'] = df.groupby((df['Coldwave_Watch']==0).cumsum())['Coldwave_Watch'].cumsum()
    df['Heatwave_streak'] = df.groupby((df['Heatwave_Watch']==0).cumsum())['Heatwave_Watch'].cumsum()
    
    return df

def create_initial_features(df):
    """
    [기본 파생 변수 생성 함수]
    날짜/시간, 체감 온도, 휴일, Lag(지연), Rolling(이동 평균) 등의 핵심 피처를 만듦
    """
    df = df.copy()
    
    # 시간 순서 정렬
    df['Datetime'] = pd.to_datetime(df[['Year','Month','Day','Hour']])
    df = df.sort_values('Datetime').reset_index(drop=True)

    T, H, V = df['Temp'], df['Humi'], df['Wind']
    
    # ── 1. 기상 체감 지표 계산 ──
    df['DI']         = 0.81*T + 0.01*H*(0.99*T - 14.3) + 46.3              # 불쾌지수
    df['Wind_Chill'] = 13.12 + 0.6215*T - 11.37*(V**0.16) + 0.3965*T*(V**0.16) # 체감온도
    df['Temp_Range'] = df['HighTemp'] - df['LowTemp']                      # 일교차

    # ── 2. 요일 및 시간 주기성 피처 ──
    df['DayOfWeek']   = df['Datetime'].dt.dayofweek
    df['Is_Weekend']  = (df['DayOfWeek'] >= 5).astype(int)
    df['Is_Monday']   = (df['DayOfWeek'] == 0).astype(int) # 월요일 더미
    df['Is_Friday']   = (df['DayOfWeek'] == 4).astype(int) # 금요일 더미
    
    # 딥러닝이 시간을 원형(주기)으로 이해하도록 sin/cos 변환
    df['Hour_sin']    = np.sin(2*np.pi*df['Hour']/24)
    df['Hour_cos']    = np.cos(2*np.pi*df['Hour']/24)
    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Day_sin']     = np.sin(2*np.pi*df['Day']/31)
    df['Day_cos']     = np.cos(2*np.pi*df['Day']/31)
    df['Weekday_sin'] = np.sin(2*np.pi*df['DayOfWeek']/7)
    df['Weekday_cos'] = np.cos(2*np.pi*df['DayOfWeek']/7)

    # ── 3. 계절 및 한국전력 부하 요금제 피처 ──
    def get_season(m):
        if m in [3,4,5]:    return 'Spring'
        elif m in [6,7,8]:  return 'Summer'
        elif m in [9,10,11]:return 'Autumn'
        else:               return 'Winter'
    df['Season'] = df['Month'].apply(get_season)

    def get_kepco_load_type(month, hour):
        # 한전 기준 경부하/중간부하/최대부하 구간 분류
        if month in [6,7,8]:
            if 13<=hour<=17:                     return 'Peak'
            elif (9<=hour<=12)or(18<=hour<=22):  return 'Mid'
            else:                                return 'Light'
        elif month in [11,12,1,2]:
            if (9<=hour<=12)or(16<=hour<=17):    return 'Peak'
            elif (13<=hour<=15)or(18<=hour<=22): return 'Mid'
            else:                                return 'Light'
        else:
            if (9<=hour<=12)or(13<=hour<=17)or(18<=hour<=22): return 'Mid'
            else:                                              return 'Light'
    df['Load_Type'] = df.apply(lambda r: get_kepco_load_type(r['Month'], r['Hour']), axis=1)

    # ── 4. 한국 공휴일 및 샌드위치 데이 피처 ──
    kr_holidays = holidays.KR(years=[2015,2016,2017,2018])
    kr_holidays[pd.to_datetime('2018-06-13').date()] = 'Local Election' # 2018년 지방선거 수동 추가
    df['Is_Holiday'] = df['Datetime'].apply(lambda x: 1 if x.date() in kr_holidays else 0)

    def check_sandwich(d):
        # 휴일 사이에 낀 평일(샌드위치 데이) 여부 판별
        if d.weekday() >= 5 or d in kr_holidays: return 0
        yesterday = d - pd.Timedelta(days=1)
        tomorrow  = d + pd.Timedelta(days=1)
        y_rest = 1 if yesterday.weekday() >= 5 or yesterday in kr_holidays else 0
        t_rest = 1 if tomorrow.weekday()  >= 5 or tomorrow  in kr_holidays else 0
        return 1 if y_rest and t_rest else 0
    df['Is_Sandwich'] = df['Datetime'].dt.date.apply(check_sandwich)

    # ── 5. 기상청 특보 및 장마 피처 결합 ──
    df.drop(columns=['Datetime'], inplace=True, errors='ignore')
    df = add_kma_warning_features(df)
    df['Is_Jangma'] = ((df['Month'].isin([6,7])) & (df['Humi'] >= 80)).astype(int) # 간이 장마 기준

    # ── 6. 시계열 지연(Lag) 및 이동 평균(Rolling) 피처 ──
    # 과거 1시간, 1일(24h), 3일(72h), 1주일(168h) 전 데이터
    for lag in [1, 24, 72, 168]:
        df[f'Temp_Lag{lag}']  = df['Temp'].shift(lag)
        df[f'DI_Lag{lag}']    = df['DI'].shift(lag)
        df[f'Humi_Lag{lag}']  = df['Humi'].shift(lag)

    # 과거 6, 12, 24시간 동안의 평균/최대/최소 기온
    for w in [6, 12, 24]:
        df[f'Temp_RollMean_{w}'] = df['Temp'].rolling(w).mean()
        df[f'Temp_RollMax_{w}']  = df['Temp'].rolling(w).max()
        df[f'Temp_RollMin_{w}']  = df['Temp'].rolling(w).min()

    # shift 과정에서 발생한 결측치는 직후 데이터로 채움
    df = df.bfill()
    return df

print('파생변수 생성 중...')
train_features = create_initial_features(raw_train_df)
test_features  = create_initial_features(raw_test_df)
print(f'피처 수: {train_features.shape[1]}')

In [ ]:
# ---------------------------------------------------------
# 3. 스케일링 및 분할
# ---------------------------------------------------------

# ── 1. 데이터 복사 및 원-핫 인코딩 (One-Hot Encoding) ──
# 원본 데이터 보호를 위해 복사본을 사용합니다.
train_data = train_features.copy()
test_data  = train_features.copy()

# 범주형 변수를 모델이 이해할 수 있도록 0과 1의 컬럼으로 펼침
train_data = pd.get_dummies(train_data, columns=['Season', 'Load_Type'])
test_data  = pd.get_dummies(test_data,  columns=['Season', 'Load_Type'])

# Train과 Test의 컬럼 종류와 순서를 완벽히 동일하게 맞춤
test_data  = test_data.reindex(columns=train_data.columns, fill_value=0)


# ── 2. 불필요한 컬럼 제거 ──
drop_cols = ['id', 'Datetime']
train_data.drop(columns=[c for c in drop_cols if c in train_data.columns], inplace=True)
test_data.drop( columns=[c for c in drop_cols if c in test_data.columns],  inplace=True)


# ── 3. Train / Validation 분할 (순차 분할) ──
n = len(train_data)
split_idx = int(n * 0.8)

train_df = train_data.iloc[:split_idx].copy()
val_df   = train_data.iloc[split_idx:].copy()
test_df  = test_data.copy()

print(f"Train 기간: {train_df['Year'].min()} ~ {train_df['Year'].max()} | {len(train_df)}행")
print(f"Val   기간: {val_df['Year'].min()}  ~ {val_df['Year'].max()}  | {len(val_df)}행")


# ── 4. 특성(X)과 타겟(y) 분리 ──
y_train = train_df['UsedPower'].values.reshape(-1, 1)
X_train = train_df.drop(columns=['UsedPower']).values

y_val = val_df['UsedPower'].values.reshape(-1, 1)
X_val = val_df.drop(columns=['UsedPower']).values

X_test = test_df.drop(columns=['UsedPower'], errors='ignore').values


# ── 5. 스케일링 (Min-Max Scaling) ──
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# 기존 로직 100% 유지: 타겟 스케일러(scaler_y)를 전체 데이터(y_all) 기준으로 학습
y_all = train_data['UsedPower'].values.reshape(-1, 1)

# X는 Train 데이터 기준으로 학습(fit) 후 변환(transform)
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled   = scaler_X.transform(X_val)
X_test_scaled  = scaler_X.transform(X_test)

# y는 전체 타겟 기준으로 학습 후 분할 지점(split_idx)에 맞게 잘라서 사용
y_train_scaled = scaler_y.fit_transform(y_all)[:split_idx].flatten()
y_val_scaled   = scaler_y.transform(y_val).flatten()

print(f"\nscaler_y 학습 범위: {scaler_y.data_min_[0]:.1f} ~ {scaler_y.data_max_[0]:.1f}")
print(f"X_train: {X_train_scaled.shape} | X_val: {X_val_scaled.shape}")

In [ ]:
# ---------------------------------------------------------
# 4. 모델 정의
# ---------------------------------------------------------

# ── 1. 활성화 함수 및 손실 함수 정의 ──
class Swish(nn.Module):
    """
    [Swish 활성화 함수] 
    기존 ReLU가 음수 값을 완전히 0으로 무시해버리는(Dying ReLU) 단점을 극복하여, 
    미세한 음수 기울기를 살려 시계열 패턴의 세밀한 학습을 도움
    """
    def forward(self, x):
        return x * torch.sigmoid(x)


def get_activation(name):
    return {'ReLU':nn.ReLU(),'LeakyReLU':nn.LeakyReLU(0.1),
            'PReLU':nn.PReLU(),'Swish':Swish()}[name]

class ScaledHuberLoss(nn.Module):
    """
    [Huber Loss 래퍼 클래스]
    MSE와 MAE의 장점을 결합하여, 
    평상시의 작은 오차는 예민하게 잡고 극단적인 이상치(극폭염/극한파)에는 덜 흔들리도록 만듦
    """
    def __init__(self, delta=0.1):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta)
    def forward(self, pred, true): return self.huber(pred, true)

# ── 2. 시계열 데이터셋 ──
class TimeSeriesDataset(Dataset):
    def __init__(self, features, target, window_size=24):
        self.features    = features.astype(np.float32)
        self.target      = target.astype(np.float32)
        self.window_size = window_size
    def __len__(self): return len(self.target) - self.window_size
    def __getitem__(self, idx):
        return (torch.tensor(self.features[idx:idx+self.window_size]),
                torch.tensor(self.target[idx+self.window_size]))

# ── 3. 하이브리드 딥러닝 아키텍처 (CNN + GRU + Attention) ──
class ImprovedHybridModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, activation_fn, num_layers=2, dropout=0.2):
        super().__init__()

        # 1. Local Feature Extraction (1D CNN)
        # 짧은 기간 동안의 날씨/전력량 변화 패턴을 빠르게 추출
        self.conv1d       = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)
        self.activation   = activation_fn
        self.bn_conv      = nn.BatchNorm1d(hidden_dim)

        # 2. Sequential Modeling (GRU)
        # CNN이 뽑아낸 특징을 바탕으로 시간의 흐름(장기적 문맥)을 기억
        self.gru          = nn.GRU(hidden_dim, hidden_dim, num_layers=num_layers,
                                   batch_first=True,
                                   dropout=dropout if num_layers > 1 else 0)
        
        # 3. Temporal Attention & Feature Gating
        # 전체 시퀀스 중 '어느 시점(시간대)'이 예측에 가장 중요한지 가중치를 부여
        self.temporal_attn = nn.Linear(hidden_dim, 1)
        self.feature_gate  = nn.Linear(hidden_dim, hidden_dim)

        # 4. Final Output
        self.dropout       = nn.Dropout(dropout)
        self.fc            = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # [CNN 단계] 파이토치 Conv1d는 (Batch, Channels, Seq) 형태를 받으므로 permute로 차원을 맞춤
        h = self.activation(self.conv1d(x.permute(0,2,1)))
        h = self.bn_conv(h).permute(0,2,1)

        # [GRU 단계]
        out, _ = self.gru(h)

        # [Attention 단계] 
        # 각 시간 스텝의 중요도(w)를 구하고, 이를 곱해서 하나의 Context Vector(ctx)로 압축
        w   = torch.softmax(self.temporal_attn(out), dim=1)
        ctx = (out * w).sum(dim=1)

        # [Gating 단계] 특정 피처의 정보를 통과시킬지 말지(0~1) 결정
        ctx = torch.sigmoid(self.feature_gate(ctx)) * ctx
        return self.fc(self.dropout(ctx)).squeeze(-1)

# ── 3. 추론 함수 (CNN + GRU + Attention) ──
def predict_full_for_submission(model, features_scaled, window_size, batch_size=256):
    """
    [전체 기간 예측 함수]
    Test 데이터 전체를 순회하며 예측값을 뽑아냄
    학습 시 잘려나간 앞부분(window_size 만큼)은 첫 번째 예측값으로 패딩(Padding)하여 
    제출용 파일의 행 개수와 완벽하게 일치시킴
    """
    model.eval()
    n       = len(features_scaled)
    dataset = TimeSeriesDataset(features_scaled, np.zeros(n), window_size=window_size)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    preds   = []
    with torch.no_grad():
        for bx, _ in loader:
            preds.extend(model(bx.to(device)).cpu().numpy())
            
    pad = np.full(window_size, preds[0])
    return np.concatenate([pad, preds])

print('모델 정의 완료')

In [ ]:
# ──────────────────────────────────────────────
# 5. 1단계 하이브리드 모델  Optuna 탐색
# ──────────────────────────────────────────────

def objective(trial, X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, scaler_y):
    """
    [Optuna 목적 함수]
    주어진 탐색 공간에서 최적의 파라미터를 찾고, 모델을 학습시켜 검증 에러(MAPE)를 반환
    """
    # ── 1. 하이퍼파라미터 탐색 공간 (Search Space) ──
    window_size = trial.suggest_categorical('window_size', [24, 48])
    hidden_dim  = trial.suggest_categorical('hidden_dim',  [64, 128])
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size  = trial.suggest_categorical('batch_size',  [32, 64])
    activation  = trial.suggest_categorical('activation', ['LeakyReLU', 'PReLU', 'Swish'])
    dropout     = trial.suggest_float('dropout', 0.1, 0.4)
    num_layers  = 1  # 1로 고정
    huber_delta = trial.suggest_float('huber_delta', 0.05, 0.5)

    # ── 2. 데이터셋 및 데이터로더 생성 ──
    train_ds = TimeSeriesDataset(X_train_scaled, y_train_scaled, window_size)
    val_ds   = TimeSeriesDataset(X_val_scaled, y_val_scaled, window_size)
    
    # 재현성(Reproducibility)을 위해 Generator 시드(42) 완벽 고정
    g = torch.Generator()
    g.manual_seed(42)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

    # ── 3. 모델, 손실 함수, 옵티마이저 초기화 ──
    model = ImprovedHybridModel(
        input_dim     = X_train_scaled.shape[1], 
        hidden_dim    = hidden_dim,
        activation_fn = get_activation(activation),
        num_layers    = num_layers, 
        dropout       = dropout
    ).to(device)

    criterion = ScaledHuberLoss(delta=huber_delta)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # [LR Scheduler] 성능 개선이 5번(patience) 정체되면 학습률을 절반으로 감소
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    best_mape = float('inf')
    no_improve = 0
    
    # ── 4. 학습 및 검증 루프 (최대 60 에포크) ──
    for epoch in range(60):
        
        # [Train Phase]
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0) # 기울기 폭발 방지
            optimizer.step()
            
        # [Validation Phase]
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for vx, vy in val_loader:
                preds.extend(model(vx.to(device)).cpu().numpy())
                trues.extend(vy.cpu().numpy())
                
        # ── 5. 스케일 복구 및 MAPE 계산 ──
        p_mw = scaler_y.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()
        t_mw = scaler_y.inverse_transform(np.array(trues).reshape(-1, 1)).flatten()
        
        # 실제값이 0보다 큰 경우만 마스킹하여 0으로 나누는 에러 방지
        mask = t_mw > 0
        mape = mean_absolute_percentage_error(t_mw[mask], p_mw[mask]) * 100
        
        scheduler.step(mape)
        
        # ── 6. Early Stopping ──
        if mape < best_mape:
            best_mape  = mape
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 10: 
                break  # 10번 연속 개선이 없으면 해당 Trial 학습 조기 종료
                
    return best_mape

# ──────────────────────────────────────────────
# ── [Optuna Study 실행] ──
# ──────────────────────────────────────────────
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

print("Optuna 하이퍼파라미터 탐색 시작")
study.optimize(
    lambda trial: objective(trial, X_train_scaled, y_train_scaled,
                            X_val_scaled, y_val_scaled, scaler_y),
    n_trials=30, timeout=5400
)

print(f'\n 탐색 완료!')
print(f'   Best Params: {study.best_params}')
print(f'   Best MAPE: {study.best_value * 100:.4f}%')

In [ ]:
# ──────────────────────────────────────────────
# 6. 1단계 하이브리드 모델 학습
# ──────────────────────────────────────────────
print('1단계 GRU 학습 시작...')

# 1. Optuna 최적 파라미터 불러오기
bp = study.best_params
bp['num_layers'] = bp.get('num_layers', 1) # 기본값 1 보장

# 2. 데이터로더 준비 (재현성을 위해 시드 42 고정)
train_ds = TimeSeriesDataset(X_train_scaled, y_train_scaled, bp['window_size'])
val_ds   = TimeSeriesDataset(X_val_scaled,   y_val_scaled,   bp['window_size'])

g = torch.Generator()
g.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=bp['batch_size'], shuffle=True, generator=g)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

# 3. 모델, 손실 함수, 옵티마이저 초기화
gru_model = ImprovedHybridModel(
    input_dim     = X_train_scaled.shape[1], 
    hidden_dim    = bp['hidden_dim'],
    activation_fn = get_activation(bp['activation']),
    num_layers    = bp['num_layers'], 
    dropout       = bp['dropout']
).to(device)

criterion = ScaledHuberLoss(delta=bp['huber_delta'])
optimizer = torch.optim.Adam(gru_model.parameters(), lr=bp['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

best_mape  = float('inf')
best_wts   = None
no_improve = 0

# 4. 본 학습 루프 (최대 150 Epoch)
for epoch in range(150):
    
    # [Train Phase]
    gru_model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        
        optimizer.zero_grad()
        loss = criterion(gru_model(bx), by)
        loss.backward()
        nn.utils.clip_grad_norm_(gru_model.parameters(), 1.0) # 기울기 폭발 방지
        optimizer.step()
        
    # [Validation Phase]
    gru_model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for vx, vy in val_loader:
            preds.extend(gru_model(vx.to(device)).cpu().numpy())
            trues.extend(vy.cpu().numpy())
            
    # 5. 스케일 복구 및 에러(MAPE) 계산
    p_mw = scaler_y.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()
    t_mw = scaler_y.inverse_transform(np.array(trues).reshape(-1, 1)).flatten()
    
    mask = t_mw > 0
    mape = mean_absolute_percentage_error(t_mw[mask], p_mw[mask])
    
    scheduler.step(mape)
    
    # 10 에포크 단위로 진행 상황 출력
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch [{epoch+1}/150] Val MAPE: {mape*100:.2f}%')
        
    # 6. 최적 가중치 저장 및 조기 종료 (Early Stopping)
    if mape < best_mape:
        best_mape  = mape
        best_wts   = copy.deepcopy(gru_model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= 15: # 15번 연속 개선 없으면 중단
            print(f'⏹ Early stop @ epoch {epoch+1}')
            break

# 7. 가장 성능이 좋았던 시점의 가중치로 모델 복구
gru_model.load_state_dict(best_wts)
print(f'\nGRU 최적 Val MAPE: {best_mape*100:.4f}%')

In [ ]:
# ──────────────────────────────────────────────
# 7. Test 데이터 예측 및 폭염 핀셋 보정
# ──────────────────────────────────────────────
test_preds_scaled = predict_full_for_submission(gru_model, X_test_scaled, bp['window_size'])
test_preds_mw     = scaler_y.inverse_transform(test_preds_scaled.reshape(-1, 1)).flatten()

test_temps = raw_test_df['Temp'].values
boosted    = test_preds_mw.copy()

# 냉방 수요 폭증 방어
boosted[(test_temps >= 30) & (test_temps < 33)] *= 1.03
boosted[test_temps >= 33]                       *= 1.05

# 예측값이 음수가 되는 것을 방지 (0 이하를 0으로 맞춤)
boosted = np.clip(boosted, 0, None)
stage1_preds = boosted

In [ ]:
# ──────────────────────────────────────────────
# 8. 2단계 LightGBM용 시계열 파생 변수 생성
# ──────────────────────────────────────────────

print('2단계 변수 구성 중...')

# ── 1. Train 데이터: 실제 과거 전력량을 이용한 피처 생성 ──
train_df_2stage = train_data.copy()

# '과거 전력 사용량' 정보 추가
train_df_2stage['UsedPower_Lag1']       = train_df_2stage['UsedPower'].shift(1)   # 직전 1시간 전
train_df_2stage['UsedPower_Lag24']      = train_df_2stage['UsedPower'].shift(24)  # 전날 동시간
train_df_2stage['UsedPower_Lag168']     = train_df_2stage['UsedPower'].shift(168) # 지난주 동시간
train_df_2stage['UsedPower_RollMean24'] = train_df_2stage['UsedPower'].shift(1).rolling(24).mean() # 과거 24시간 평균
train_df_2stage['UsedPower_RollStd24']  = train_df_2stage['UsedPower'].shift(1).rolling(24).std()  # 과거 24시간 변동성

# shift로 인해 앞부분에 발생한 결측치(NaN) 행을 깔끔하게 제거
train_df_2stage = train_df_2stage.dropna().reset_index(drop=True)

# ── 2. Test 데이터: 1단계 예측값(stage1_preds)을 이용한 피처 생성 ──
test_df_2stage = create_initial_features(raw_test_df)
test_df_2stage = pd.get_dummies(test_df_2stage, columns=['Season','Load_Type'])
test_df_2stage = test_df_2stage.reindex(columns=train_data.columns, fill_value=0)

for col in ['id','Datetime']:
    if col in test_df_2stage.columns:
        test_df_2stage.drop(columns=[col], inplace=True)

# Test 구간의 과거 전력량은 정답이 없으므로, 1단계 딥러닝이 예측한 값을 대신 사용
train_actuals = train_data['UsedPower'].values
test_len      = len(test_df_2stage)

test_lag1        = np.zeros(test_len)
test_lag1[0]     = train_actuals[-1]
test_lag1[1:]    = stage1_preds[:-1]

test_lag24       = np.zeros(test_len)
test_lag24[:24]  = train_actuals[-24:]
test_lag24[24:]  = stage1_preds[:-24]

test_lag168        = np.zeros(test_len)
test_lag168[:168]  = train_actuals[-168:]
test_lag168[168:]  = stage1_preds[:-168]

# [Rolling] Train 마지막 24시간과 Test 예측값을 하나로 붙여 연속적인 24시간 이동평균/표준편차를 구함
combined       = np.concatenate([train_actuals[-24:], stage1_preds])
cs             = pd.Series(combined)
test_roll_mean = cs.shift(1).rolling(24).mean().values[24:]
test_roll_std  = cs.shift(1).rolling(24).std().values[24:]

test_df_2stage['UsedPower_Lag1']       = test_lag1
test_df_2stage['UsedPower_Lag24']      = test_lag24
test_df_2stage['UsedPower_Lag168']     = test_lag168
test_df_2stage['UsedPower_RollMean24'] = test_roll_mean
test_df_2stage['UsedPower_RollStd24']  = test_roll_std

# ── 3. 모델에 투입할 최종 피처(Features) 리스트 확정 ──
features = [
    # [시간 주기 및 요일] 
    'Hour', 'Hour_sin', 'Hour_cos',
    'Month_sin', 'Month_cos', 'Day_sin', 'Day_cos',
    'Weekday_sin', 'Weekday_cos',
    'DayOfWeek', 'Is_Weekend', 'Is_Holiday', 'Is_Sandwich',
    
    # [기본 기상 지표]
    'Temp', 'Wind_Chill', 'Temp_Range', 'Humi', 'Wind', 'HeatIndex',
    'LowTemp_prev1d', 'LowTemp_drop_1d', 'Below_avg3', 'MaxHeatIndex_daily',
    
    # [기상청 특보 및 장마]
    'Coldwave_Level', 'Heatwave_Level', 
    'Coldwave_streak', 'Heatwave_streak', 'Is_Jangma',
    
    # [기상 지표 지연(Lag)] 
    'Temp_Lag1', 'Temp_Lag24', 'Temp_Lag72', 'Temp_Lag168',
    'Humi_Lag1', 'Humi_Lag24', 'Humi_Lag72', 'Humi_Lag168',
    'HeatIndex_Lag1', 'HeatIndex_Lag24', 'HeatIndex_Lag72', 'HeatIndex_Lag168',
    
    # [기온 이동 평균/최대/최소(Rolling)]
    'Temp_RollMean_6', 'Temp_RollMax_6', 'Temp_RollMin_6',
    'Temp_RollMean_12', 'Temp_RollMax_12', 'Temp_RollMin_12',
    'Temp_RollMean_24', 'Temp_RollMax_24', 'Temp_RollMin_24',
    
    # [1단계 전력량 지연 및 이동평균 (Target Feature)]
    'UsedPower_Lag1', 'UsedPower_Lag24', 'UsedPower_Lag168',
    'UsedPower_RollMean24', 'UsedPower_RollStd24',
]

# ── 4. 학습/테스트 데이터 세트 최종 구성 ──
X_train_final = train_df_2stage[features]
y_train_final = train_df_2stage['UsedPower']
X_test_final  = test_df_2stage[features]

print(f'X_train_final: {X_train_final.shape}')
print(f'X_test_final : {X_test_final.shape}')
print(f'NaN 확인     : {X_test_final.isna().sum().sum()}')

In [ ]:
# ──────────────────────────────────────────────
# 9. 2단계 LightGBM Optuna
# ──────────────────────────────────────────────
print('Stage2 LightGBM Optuna 탐색 시작...')

# ── 1. 2단계 검증용 데이터 분할 (Time-Series Split) ──
split_2 = int(len(X_train_final) * 0.8)
X_tr2 = X_train_final.iloc[:split_2]; y_tr2 = y_train_final.iloc[:split_2]
X_vl2 = X_train_final.iloc[split_2:]; y_vl2 = y_train_final.iloc[split_2:]

# ── 2. Optuna 목적 함수 정의 ──
def lgb_objective(trial):
    """
    주어진 탐색 공간에서 최적의 LightGBM 트리 설정값을 찾아내고,
    검증 데이터(X_vl2)에 대한 MAPE를 반환합니다.
    """
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 500),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves'      : trial.suggest_int('num_leaves', 7, 31),
        'max_depth'       : trial.suggest_int('max_depth', 3, 6),
        'subsample'       : trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr2, y_tr2,
                eval_set=[(X_vl2, y_vl2)],
                callbacks=[lgb.early_stopping(30, verbose=False)])
    return mean_absolute_percentage_error(y_vl2, model.predict(X_vl2)) * 100

# ── 3. Optuna Study 실행 ──
study_stage2 = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_stage2.optimize(lgb_objective, n_trials=50)
print(f'   Best Params: {study_stage2.best_params}')
print(f'   best MAPE: {study_stage2.best_value:.4f}%')